# Wind Load Calc — Sign on a Vandal Protection Fence on a Bridge Railing

A 3 ft × 5 ft solid metal sign clamped low — bottom of panel at the top of the
parapet — on a post of an **ODOT Vandal Protection Fence** (SCD VPF-1-24 /
legacy VPF-1-90) mounted on a concrete bridge railing.
We chase the wind force through the whole load path and report demand/capacity
at every link:

1. **Wind load** — AASHTO *LRFD Specifications for Structural Supports for
   Highway Signs, Luminaires, and Traffic Signals* (LRFDLTS-1), Article 3.8
2. **Sign-to-fence connection** — U-bolt clamps (LTS 5.15 → AISC bolt shear)
3. **Fence post** — cantilever flexure (LTS Section 5)
4. **Fabric-to-post connection** — tension bands per VPF general note (7)
5. **Anchor bolts** — adhesive anchors per VPF note (5) →
   `civilpy.structural.concrete.AnchorBolts` (ACI 318-19 Ch. 17, which is where
   LTS 5.16.3 and the VPF drawings both send anchorage design)

Fence hardware is taken from the drawings, not assumed: **6'-0" straight fence**
(VPF-1-24 — the 8'-0" straight option existed on VPF-1-90 but the current
standard is 6'), 1-in 11-ga mesh, one tension band per foot of fabric, and the
standard base plates.  Load combination: **Extreme I** — 1.0·W with the 700-yr
MRI wind map, 1.1/0.9·DC (LTS Table 3.4-1, Table 3.8-1).

## Geometry and hardware (from the SCDs)

| item | value | source |
|---|---|---|
| fence fabric height, $H_f$ | 6'-0" | VPF-1-24 post section PS-2 |
| fabric | 1-in diamond mesh, 0.120-in (11-ga) wire | VPF note (16) |
| line/end posts | 2.880-in OD Grade 2 pipe, Fy = 50 ksi, 4.64 lb/ft | VPF note (1) |
| rails (top/line/bottom) | 1.660-in OD Grade 2 pipe, Fy = 50 ksi | VPF note (2) |
| post spacing | 10'-0" max with BP-1 (5'-0" with BP-2) | VPF-1-24 sheet 2 |
| tension bands | ⅛×1-in, one per ft of fabric, ⅜-in bolts | VPF note (7) |
| base plate (primary case) | VPF-1-90 BP-1: 8½×13×1-in flat plate | VPF-1-90 sheet 3 |
| anchors (primary case) | 4 × ½-in adhesive rods, 7-in min embed; analyzed as **F1554 Gr 36** per current standard | VPF-1-90 notes (4)/(5); VPF-1-24 |
| adhesive | Hilti HIT-HY 200 (first listed approved product) | ICC-ES ESR-3187 |
| railing | BR-type parapet, 12-in top width, f′c ≥ 4,000 psi | VPF-1-90 sheet 2, note (22) |
| sign | 3'T × 5'W solid panel, **bottom at the top of the parapet**, centered on one post | design case |
| site | Ohio — V(700-yr) = 115 mph, V(10-yr, Fig. 3.8-4) = 76 mph | LTS Figs. 3.8-1b/3.8-4 |

Risk category is *Typical* (a failed post/sign could drop into the travelway),
so Table 3.8-1 gives MRI = 700 yrs; the "roadside sign supports: 10-yr" relief
does not apply.  VPF note (22) requires a **special design** when f′c < 4,000
psi or the fence midpoint sits more than 50 ft above the surrounding terrain —
this calc *is* that special-design check for the sign retrofit; we keep the
fence below the 50-ft trigger so Kz rides its 16-ft floor.

The current VPF-1-24 base plates are all bent (wrap-around) plates with ⅝-in
F1554 Gr 36 rods (verticals 6-in embed, horizontal 4-in); the flat 8½×13 plate
above is the legacy VPF-1-90 detail found on existing bridges — the likely
candidates for a sign retrofit — and is run as the primary case.  The
wrap-plate interface is checked second, against ODOT's own published design
load.

In [1]:
import math
import numpy as np
from civilpy.general import units

# --- site / spec inputs -------------------------------------------------
V   = 115.0        # mph, 3-s gust, 700-yr MRI (LTS Fig. 3.8-1b, Ohio)
V10 = 76.0         # mph, 10-yr MRI service wind (LTS Fig. 3.8-4)

# --- fence geometry (VPF-1-24) -------------------------------------------
H_fabric = 6.0            # ft, fabric height above the base plate
s_post   = 10.0           # ft, max post spacing with BP-1
d_post   = 2.880 / 12.0   # ft, post OD
t_post   = 0.160          # in, Grade 2 pipe wall (4.64 plf checks below)
d_rail   = 1.660 / 12.0   # ft, rail OD
n_rails  = 3              # top, line (mid-height), bottom

# fabric solidity: 1-in diamond mesh of 0.120-in wire has two wire runs
# crossing each 1-in cell -> projected solid fraction ~ 2*d/mesh
wire_dia, mesh = 0.120, 1.0
solidity = 2 * wire_dia / mesh
print(f"fabric solidity (1-in mesh, 11-ga) ~ {solidity:.2f}")

# --- sign: mounted LOW, bottom of panel at the top of the parapet ---------
sign_w, sign_h = 5.0, 3.0            # ft
A_sign = sign_w * sign_h
z_sign_cg = sign_h / 2.0             # centroid above base plate, ft
z_sign_cg_top = H_fabric - sign_h/2  # top-mounted alternative, for sensitivity
print(f"sign {A_sign:.0f} ft^2, centroid {z_sign_cg:.1f} ft above the base plate (bottom-mounted)")

fabric solidity (1-in mesh, 11-ga) ~ 0.24
sign 15 ft^2, centroid 1.5 ft above the base plate (bottom-mounted)


## Step 1 — Design wind pressure (LTS Article 3.8)

$$P_z = 0.00256\,K_z\,K_d\,G\,V^2\,C_d \quad \text{(psf, Eq. 3.8.1-1)}$$

- $K_z$ — Eq. 3.8.4-1 (Exposure C: $\alpha=9.5$, $z_g=900$ ft, $z\ge16$ ft)
- $K_d$ — Table 3.8.5-1: round pole → 0.95
- $G$ — Art. 3.8.6: 1.14 minimum
- $C_d$ — Table 3.8.7-1: sign panel interpolated on $L_{sign}/W_{sign}$;
  cylindrical members by the $C_vVd$ regime ($C_v=0.8$ at the extreme limit
  state).  Chain-link fabric is not tabulated — note *a* permits established
  values for untabulated shapes, so the fabric is treated as its net (solid)
  area with $C_d = 1.2$ (CLFMI wind-load-guide practice).

In [2]:
def Kz(z_ft):
    """LTS Eq. 3.8.4-1, Exposure C, with the 16-ft floor."""
    z = max(z_ft, 16.0)
    return 2.0 * (z / 900.0) ** (2.0 / 9.5)

Kd = 0.95      # Table 3.8.5-1, round pole
G  = 1.14      # Art. 3.8.6

def Cd_cylinder(d_ft, Cv=0.8):
    """Table 3.8.7-1, single cylindrical member."""
    cvvd = Cv * V * d_ft
    if cvvd <= 39.0:
        return 1.10
    if cvvd < 78.0:
        return 129.0 / cvvd**1.3
    return 0.45

# sign panel Cd by aspect ratio (Table 3.8.7-1)
ratios, cds = [1.0, 2.0, 5.0, 10.0, 15.0], [1.12, 1.19, 1.20, 1.23, 1.30]
Cd_sign   = float(np.interp(sign_w / sign_h, ratios, cds))
Cd_fabric = 1.2
Cd_post   = Cd_cylinder(d_post)
Cd_rail   = Cd_cylinder(d_rail)

kz = Kz(16.0)                          # fence top well under the 16-ft floor height
q  = 0.00256 * kz * Kd * G * V**2      # psf per unit Cd
print(f"Kz = {kz:.3f} ->  q = {q:.1f} psf per unit Cd")
print(f"Cd: sign {Cd_sign:.3f} (L/W = {sign_w/sign_h:.2f}) | fabric 1.2 | post {Cd_post:.2f} | rail {Cd_rail:.2f}")
print(f"Pz: sign {q*Cd_sign:.1f} psf | fabric (net) {q*Cd_fabric:.1f} psf | members {q*Cd_post:.1f} psf")

Kz = 0.856 ->  q = 31.4 psf per unit Cd
Cd: sign 1.167 (L/W = 1.67) | fabric 1.2 | post 1.10 | rail 1.10
Pz: sign 36.6 psf | fabric (net) 37.7 psf | members 34.5 psf


In [3]:
# Factored member-end forces, Extreme I (gamma_W = 1.0) --------------------
# The sign shadows its own patch of fabric; rails average out at mid-height.
F_sign   = q * Cd_sign   * A_sign
A_fab_net_sign = (s_post * H_fabric - A_sign) * solidity      # sign post bay
A_fab_net_line = (s_post * H_fabric) * solidity               # plain bay
F_fab_s  = q * Cd_fabric * A_fab_net_sign
F_fab_l  = q * Cd_fabric * A_fab_net_line
F_post   = q * Cd_post   * (d_post * H_fabric)
F_rails  = q * Cd_rail   * (n_rails * s_post * d_rail)

# sign post
V_sign_post = F_sign + F_fab_s + F_post + F_rails
M_sign_post = (F_sign * z_sign_cg
               + (F_fab_s + F_post + F_rails) * H_fabric / 2) * 12   # lb-in

# plain line post
V_line = F_fab_l + F_post + F_rails
M_line = V_line * H_fabric / 2 * 12

print(f"F_sign = {F_sign:5.0f} lb @ {z_sign_cg:.1f} ft   F_fabric = {F_fab_s:.0f} lb   "
      f"F_post = {F_post:.0f} lb   F_rails = {F_rails:.0f} lb")
print(f"sign post:  V = {V_sign_post:5.0f} lb   M = {M_sign_post/1000:5.1f} kip-in")
print(f"line post:  V = {V_line:5.0f} lb   M = {M_line/1000:5.1f} kip-in")

F_sign =   549 lb @ 1.5 ft   F_fabric = 407 lb   F_post = 50 lb   F_rails = 143 lb
sign post:  V =  1149 lb   M =  31.5 kip-in
line post:  V =   736 lb   M =  26.5 kip-in


## Step 2 — Sign-to-fence connection

The panel is clamped to the sign post with **two ⅜-in U-bolts** (4 shear legs),
matching the fence's own ⅜-in hardware (VPF notes 7/8).  LTS 5.15 sends bolted
connections to AISC: $\phi R_n = 0.75\,F_{nv}A_b$, $F_{nv}=27$ ksi (A307-class,
threads included).

In [4]:
n_legs = 4
V_leg = F_sign / n_legs
A_b38 = math.pi * 0.375**2 / 4
phiRn_leg = 0.75 * 27_000 * A_b38
dc_ubolt = V_leg / phiRn_leg
print(f"shear/leg = {V_leg:.0f} lb  vs  phiRn = {phiRn_leg:.0f} lb  ->  D/C = {dc_ubolt:.2f}")

shear/leg = 137 lb  vs  phiRn = 2237 lb  ->  D/C = 0.06


## Step 3 — Fence post flexure

Posts are the drawing's actual pipe — 2.880-in OD × 0.160-in wall Grade 2
(Fy = 50 ksi, 4.64 plf) — *not* Sch 40, so we compute section properties
directly.  Per LTS Table 5.8.2-1 a round tube with
$D/t \le \lambda_p = 0.07\,E/F_y$ is **compact** and takes the full plastic
moment, $\phi M_n = 0.9\,F_y\,Z$ with $Z = (OD^3 - ID^3)/6$ for a hollow round
— about a third more capacity than the elastic $S$.

In [5]:
Fy_post = 50_000.0     # psi, Grade 2 pipe per VPF note (1)

OD, t = 2.880, t_post
ID = OD - 2*t
A_p = math.pi/4 * (OD**2 - ID**2)
I_p = math.pi/64 * (OD**4 - ID**4)
S_p = I_p / (OD/2)
Z_p = (OD**3 - ID**3) / 6
print(f"post section: A = {A_p:.3f} in^2, I = {I_p:.3f} in^4, S = {S_p:.3f} in^3, "
      f"Z = {Z_p:.3f} in^3, weight = {A_p*3.4:.2f} plf (drawing says 4.64)")

# compactness (LTS Table 5.8.2-1, round tube in flexure)
E = 29_000_000.0
lam, lam_p = OD / t, 0.07 * E / Fy_post
print(f"D/t = {lam:.1f} vs lambda_p = {lam_p:.1f} -> "
      + ("compact: Mn = Mp = Fy*Z" if lam <= lam_p else "NONCOMPACT — Z not permitted"))

phiMn_post = 0.9 * Fy_post * Z_p
dc_line  = M_line / phiMn_post
dc_signp = M_sign_post / phiMn_post
print(f"\nline post:  M = {M_line/1000:5.1f} kip-in vs phiMn = {phiMn_post/1000:.1f} kip-in -> D/C = {dc_line:.2f}")
print(f"sign post:  M = {M_sign_post/1000:5.1f} kip-in vs phiMn = {phiMn_post/1000:.1f} kip-in -> D/C = {dc_signp:.2f}"
      + ("   <-- FAILS" if dc_signp > 1 else ""))

post section: A = 1.367 in^2, I = 1.269 in^4, S = 0.881 in^3, Z = 1.185 in^3, weight = 4.65 plf (drawing says 4.64)
D/t = 18.0 vs lambda_p = 40.6 -> compact: Mn = Mp = Fy*Z

line post:  M =  26.5 kip-in vs phiMn = 53.3 kip-in -> D/C = 0.50
sign post:  M =  31.5 kip-in vs phiMn = 53.3 kip-in -> D/C = 0.59


Mounting the sign **low keeps the standard post working** — the panel adds
force but almost no lever arm.  Mounting height is the single biggest lever
in this whole calc, so check the alternative before anyone field-relocates
the sign upward:

In [6]:
# sensitivity: same sign ridden up to the top of the fence -----------------
M_top = (F_sign * z_sign_cg_top
         + (F_fab_s + F_post + F_rails) * H_fabric / 2) * 12
dc_top = M_top / phiMn_post
print(f"sign at TOP of fence: M = {M_top/1000:.1f} kip-in -> std post D/C = {dc_top:.2f}"
      + ("  <-- FAILS" if dc_top > 1 else "  (no strength margin left)"))

from civilpy.structural.steel import SteelSection
big = SteelSection("Pipe4SCH40")
Z_big = big.Z_x.magnitude              # NPS 4 Sch 40 is compact at Fy = 35 ksi too
phiMn_big = 0.9 * 35_000 * Z_big
dc_big = M_top / phiMn_big
print(f"  (an NPS 4 post would restore real flexural margin: Pipe4SCH40 phiMn = "
      f"{phiMn_big/1000:.0f} kip-in -> D/C = {dc_big:.2f})")

# Service I deflection at the 10-yr wind (LTS Fig. 3.8-4), std post, low sign
z_res = M_sign_post / V_sign_post / 12          # resultant height, ft
P_serv = V_sign_post * (V10 / V) ** 2
delta = P_serv * (z_res*12)**3 / (3 * E * I_p)
print(f"10-yr wind: {P_serv:.0f} lb at {z_res:.1f} ft -> std-post tip deflection ~ {delta:.2f} in")

sign at TOP of fence: M = 51.3 kip-in -> std post D/C = 0.96  (no strength margin left)
  (an NPS 4 post would restore real flexural margin: Pipe4SCH40 phiMn = 128 kip-in -> D/C = 0.40)
10-yr wind: 502 lb at 2.3 ft -> std-post tip deflection ~ 0.09 in


### Combined forces — LTS 5.12.1

Bending never acts alone: Article 5.12.1 requires the interaction of axial
load, flexure, shear, and torsion.  With the sign centered on the post,
$T_u = 0$, so torsion and shear drop out of Eq. 5.12.1-1, and with
$P_u/P_r < 0.2$ the governing form is **Eq. 5.12.1-3**:

$$\frac{P_u}{2P_r} + \frac{B\,M_u}{M_r} \le 1.0$$

$P_u$ is just the factored self-weight standing on the base plate (1.1·DC,
Extreme I), $P_r$ the compression resistance of the post as a cantilever
column (LTS 5.10, K = 2.1), and $B$ the second-order magnifier.

In [7]:
# dead load carried by the sign post (tributary) ---------------------------
w_sign_panel = 1.8 * A_sign            # psf, 0.125-in aluminum panel (ODOT sign blank)
DL = (4.64 * H_fabric                  # post pipe, 4.64 plf per VPF note (1)
      + 2.27 * n_rails * s_post        # 1.660-OD rails ~2.27 plf, tributary span
      + 0.8 * s_post * H_fabric        # 1-in 11-ga fabric ~0.8 psf
      + w_sign_panel)
Pu = 1.1 * DL                          # Extreme I max DC factor
print(f"dead load = {DL:.0f} lb -> Pu = {Pu:.0f} lb")

# compression resistance, cantilever column (LTS 5.10) ---------------------
K, L = 2.1, H_fabric * 12
r_gyr = math.sqrt(I_p / A_p)
klr = K * L / r_gyr
Fe = math.pi**2 * E / klr**2
Fcr = (0.658 ** (Fy_post / Fe)) * Fy_post if Fy_post / Fe <= 2.25 else 0.877 * Fe
Pr = 0.9 * Fcr * A_p
print(f"KL/r = {klr:.0f}, Fe = {Fe/1000:.1f} ksi -> Fcr = {Fcr/1000:.1f} ksi, "
      f"Pr = {Pr:.0f} lb   (Pu/Pr = {Pu/Pr:.3f} < 0.2 -> Eq. 5.12.1-3)")

# second-order magnifier and interaction ------------------------------------
Pe = math.pi**2 * E * I_p / (K * L)**2
B_mag = 1.0 / (1.0 - Pu / Pe)
ix_low = Pu/(2*Pr) + B_mag * M_sign_post / phiMn_post
ix_top = Pu/(2*Pr) + B_mag * M_top      / phiMn_post
print(f"B = {B_mag:.3f}")
print(f"interaction, sign low (design): {ix_low:.2f}"
      + ("  <-- FAILS" if ix_low > 1 else "  OK"))
print(f"interaction, sign at fence top: {ix_top:.2f}"
      + ("  <-- FAILS" if ix_top > 1 else ""))

dead load = 171 lb -> Pu = 188 lb
KL/r = 157, Fe = 11.6 ksi -> Fcr = 10.2 ksi, Pr = 12538 lb   (Pu/Pr = 0.015 < 0.2 -> Eq. 5.12.1-3)
B = 1.012
interaction, sign low (design): 0.60  OK
interaction, sign at fence top: 0.98


Self-weight adds ~1% — flexure was always the story on a fence post — but
the check is now on paper, and the same cell is where a heavier attachment
(a cabinet, a camera) would start to matter.

## Step 4 — Fabric-to-post connection

Per VPF note (7): **one ⅛×1-in tension band per foot of fabric height** — six
bands on the 6-ft fence — each with a ⅜-in galvanized bolt in single shear.
(Fabric ties, note 11, carry the same rhythm on line posts.)

In [8]:
n_bands = int(H_fabric)          # one per foot of fabric height
V_band = F_fab_s / n_bands
phiRn_band = 0.75 * 27_000 * A_b38
dc_band = V_band / phiRn_band
print(f"{n_bands} bands -> {V_band:.0f} lb/bolt vs phiRn = {phiRn_band:.0f} lb -> D/C = {dc_band:.2f}")
print(f"(average net fabric pressure is only {F_fab_s/(s_post*H_fabric):.1f} psf gross — "
      "the 11-ga fabric itself is nowhere near its breaking strength)")

6 bands -> 68 lb/bolt vs phiRn = 2237 lb -> D/C = 0.03
(average net fabric pressure is only 6.8 psf gross — the 11-ga fabric itself is nowhere near its breaking strength)


## Step 5 — Anchor rods (VPF-1-90 BP-1 flat plate, adhesive anchors)

The legacy flat plate: **8½ × 13 × 1-in**, four **½-in threaded rods** set with
adhesive at **7-in min embedment** into the 12-in-wide BR parapet top.  Rod
columns sit 3 in and 7 in from the back face (4-in gage across the parapet,
10-in rows along it); the post is offset ~2½ in from the back edge.
VPF-1-90 called out A193 B7 rod stock, but the current VPF-1-24 standard
specifies **ASTM F1554 Grade 36** (Fy = 36 ksi, Fu = 58 ksi) — the analysis
uses Gr 36, which matches new work and bounds any legacy B7 install from
below.

Adhesive design values from **ICC-ES ESR-3187** (Hilti HIT-HY 200, the first
approved product on both VPF drawings), hammer-drilled, dry, temp range A:

| parameter | value | ESR table |
|---|---|---|
| τ_k,cr / τ_k,uncr (½-in rod) | 1,135 / 2,220 psi | Table 14 |
| k_c (cracked / uncracked) | 17 / 24 | Table 12 |
| φ tension / shear, concrete modes (Cond. B) | 0.65 / 0.70 | Table 12 |
| ½-in F1554 Gr 36 rod: N_sa / V_sa | 8,230 / 4,940 lb (φ = 0.75/0.65) | Table 10 |
| permitted h_ef, ½-in | 2¾ – 10 in | Table 12 |

The parapet top is treated as **cracked** concrete (conservative, and the VPF
notes require products qualified for cracked concrete).  The base moment
resolves into a couple across the 4-in bolt gage; wind can blow either way, so
the tension pair is taken at the worst column — the back column with just 3 in
of edge distance.

**Code-edition note:** LTS 5.16.3 (1st Ed.) formally delegates anchorage
design to **ACI 318-11 Appendix D** — the very edition in which adhesive
anchors entered the code.  ACI 318-19 Chapter 17, which `AnchorBolts` cites,
is those same provisions renumbered (App. D bond model D.5.5 → 17.6.5, same
uniform-bond equations and ψ factors for every limit state checked here), so
the results are identical under either edition.  The modern citation is kept
because it is what ESR-3187 anchors its published design values to.

In [9]:
gage = 4.0                                  # in, across the parapet
T_pair = M_sign_post / gage                 # lb on the 2-rod tension column
V_group = V_sign_post
print(f"tension couple T = M/gage = {M_sign_post/1000:.1f} kip-in / {gage:.0f} in = "
      f"{T_pair:.0f} lb on the tension column ({T_pair/2:.0f} lb/rod)")
print(f"group shear V = {V_group:.0f} lb ({V_group/4:.0f} lb/rod)")

tension couple T = M/gage = 31.5 kip-in / 4 in = 7871 lb on the tension column (3936 lb/rod)
group shear V = 1149 lb (287 lb/rod)


In [10]:
from civilpy.structural.concrete import AnchorBolts

# projected breakout area of the tension column, actual parapet geometry
# (ACI 17.6.2.1.1): edges only across the 12-in width; barrier runs long.
h_ef, c_back, c_road, s_row = 7.0, 3.0, 9.0, 10.0
x_ext = min(c_back, 1.5*h_ef) + min(c_road, 1.5*h_ef)         # across parapet
y_ext = 1.5*h_ef + s_row + 1.5*h_ef                            # along parapet
A_Nc_hand = x_ext * y_ext
print(f"hand A_Nc = {x_ext:.1f} x {y_ext:.1f} = {A_Nc_hand:.0f} in^2 "
      f"(vs A_Nco = {(3*h_ef)**2:.0f} per anchor)")

anchors = AnchorBolts(
    f_c=4000.0, h_a=30.0,                      # parapet depth below the top
    d_a=0.5, h_ef=h_ef,
    f_ya=36_000.0, f_uta=58_000.0,             # ASTM F1554 Gr 36 (ESR-3187 Table 10)
    n_x=1, n_y=2, s_x=0.0, s_y=s_row,          # the tension column
    c_a1=c_back, c_a2=100.0,
    A_Nc=A_Nc_hand,
    anchor_type="adhesive",
    tau_cr=1135.0, tau_uncr=2220.0,            # ESR-3187 Table 14, 1/2-in rod
    is_cracked=True, has_supp_reinf=False,
    N_ua=T_pair, V_ua=V_group / 2.0,           # column's share of shear
    shear_direction="perpendicular",
)
print(anchors.summary())

hand A_Nc = 12.0 x 31.0 = 372 in^2 (vs A_Nco = 441 per anchor)
-----------------------------------------------------------------------------------------------------
Limit State                         Ref                           φSn (kip)     Demand     DCR Status
-----------------------------------------------------------------------------------------------------
Steel strength (tension)            ACI 318-19 Eq. 17.6.1.2           12.35       7.87   0.638     OK
Concrete breakout (tension)         ACI 318-19 Eq. 17.6.2.1b           9.24       7.87   0.852     OK
Bond strength (tension)             ACI 318-19 Eq. 17.6.5.1.1b         7.13       7.87   1.104     NG
Steel strength (shear)              ACI 318-19 Eq. 17.7.1.2b           6.42       0.57   0.090     OK
Concrete breakout (shear)           ACI 318-19 Eq. 17.7.2.1b           3.64       0.57   0.158     OK
Pryout (shear)                      ACI 318-19 Eq. 17.7.3.1           18.48       0.57   0.031     OK
Tension-shear inter

*(The `AnchorBolts` default projected areas clip to `c_a_min` on **all**
sides — correct for a pedestal, conservative on a long parapet — so the
tension-breakout area above is hand-computed per ACI 17.6.2.1.1 and passed as
an override.  Bond influence areas inside the class carry the same
conservatism; the printed bond DCR is therefore a lower bound on capacity.)*

Run the same anchorage under the **bare fence** (no sign) for comparison:

In [11]:
T_pair_fence = M_line / gage
anchors_fence = AnchorBolts(
    f_c=4000.0, h_a=30.0, d_a=0.5, h_ef=h_ef,
    f_ya=36_000.0, f_uta=58_000.0,
    n_x=1, n_y=2, s_x=0.0, s_y=s_row, c_a1=c_back, c_a2=100.0,
    A_Nc=A_Nc_hand, anchor_type="adhesive",
    tau_cr=1135.0, tau_uncr=2220.0, is_cracked=True,
    N_ua=T_pair_fence, V_ua=V_line / 2.0,
    shear_direction="perpendicular",
)
worst_sign  = max(r.dcr for r in anchors.check_all().values())
worst_fence = max(r.dcr for r in anchors_fence.check_all().values())
print(f"bare fence  (T = {T_pair_fence:5.0f} lb): worst anchor DCR = {worst_fence:.2f}")
print(f"fence + sign (T = {T_pair:5.0f} lb): worst anchor DCR = {worst_sign:.2f}")

bare fence  (T =  6620 lb): worst anchor DCR = 0.93
fence + sign (T =  7871 lb): worst anchor DCR = 1.10


**The governing link is the concrete side of the anchorage.**  Even with the
sign mounted low, the bond limit state edges past 1.0 (on a bond-influence
area the class clips conservatively), while even the soft Gr 36 rods keep
steel margin.  Adding or upsizing rods does nothing; concrete governs —
which is also the LTS 5.16.3 ductility complaint: the steel cannot reach
its strength before the concrete lets go.  (Gr 36 at least *narrows* that
gap versus the legacy B7 callout — one reason the current standard switched.)

> **Assumptions:**
>
> - *Adhesive verification* — the bond strengths above are ESR-3187 values
>   current at design time; ICC-ES reports are reissued frequently and
>   formulations change.  Did not verify: that the adhesive anchor system
>   used meets or exceeds the assumed characteristic bond strengths
>   (τ_k,cr = 1,135 psi cracked, ½-in rod) per the active ICC-ES evaluation
>   report at the time of construction."*
>
> - *Base plate prying:* the anchor tensions above assume a rigid
>   plate; a flexible plate levers extra tension into the rods (prying
>   action, AISC 360 §9 / Design Guide 1).  The VPF-1-90 plate is 1 in
>   thick on a ~2½-in cantilever from the post wall to the rod line —
>   almost certainly rigid.

## Cross-check — the wrap-around plate and ODOT's published design load

VPF-1-90's general notes state the factored design load for the two-anchor
**horizontal** connection of the wrap-around BP-3 plate: **7.1 kips tension +
1.4 kips shear**.  That is ODOT telling us the demand envelope the standard
fence was designed to.  Resolving our post base moment into the wrap-plate's
horizontal pair (bearing at the top corner, anchors ~5 in below it):

In [12]:
lever = 5.0     # in, top-corner bearing to horizontal anchor line (BP-3 end view, MIN.)
T_wrap_fence = M_line / lever
T_wrap_sign  = M_sign_post / lever
print(f"bare fence:   T = {T_wrap_fence/1000:.1f} kips vs ODOT design envelope 7.1 kips "
      f"({T_wrap_fence/7100:.0%})")
print(f"fence + sign: T = {T_wrap_sign/1000:.1f} kips vs 7.1 kips ({T_wrap_sign/7100:.0%})"
      + ("   <-- exceeds the standard detail's design basis" if T_wrap_sign > 7100 else ""))

bare fence:   T = 5.3 kips vs ODOT design envelope 7.1 kips (75%)
fence + sign: T = 6.3 kips vs 7.1 kips (89%)


The bare fence sits inside ODOT's envelope — consistent with the drawing
being a standard — and the sign pushes past it.  Same conclusion from the
other direction, using ODOT's own number instead of ours.

## Load-path summary

In [13]:
import pandas as pd

rows = [
    ("1. wind pressure (LTS 3.8)",        f"{q*Cd_sign:.1f} psf on sign", "-", None),
    ("2. sign U-bolt clamps",             f"{V_leg:.0f} lb/leg",  f"{phiRn_leg:.0f} lb", dc_ubolt),
    ("3a. line post (bare fence)",        f"{M_line/1000:.1f} k-in", f"{phiMn_post/1000:.1f} k-in", dc_line),
    ("3b. sign post, sign low (design)",  f"{M_sign_post/1000:.1f} k-in", f"{phiMn_post/1000:.1f} k-in", dc_signp),
    ("3c. sign post if sign at fence top",f"{M_top/1000:.1f} k-in", f"{phiMn_post/1000:.1f} k-in", dc_top),
    ("3d. post P+M interaction (5.12.1-3)", "sign low / at top", "1.0", max(ix_low, ix_top)),
    ("4. fabric tension-band bolts",      f"{V_band:.0f} lb", f"{phiRn_band:.0f} lb", dc_band),
    ("5a. anchors, bare fence",           f"{T_pair_fence:.0f} lb", "concrete governs", worst_fence),
    ("5b. anchors, fence + low sign",     f"{T_pair:.0f} lb", "concrete governs", worst_sign),
    ("5c. wrap plate vs ODOT 7.1k",       f"{T_wrap_sign/1000:.1f} kips", "7.1 kips", T_wrap_sign/7100),
    ("5d. wrap plate if sign at top",     f"{M_top/lever/1000:.1f} kips", "7.1 kips", M_top/lever/7100),
]
df = pd.DataFrame(rows, columns=["load-path link", "demand", "capacity", "D/C"])
df["status"] = df["D/C"].apply(lambda x: "-" if pd.isna(x) else ("OK" if x <= 1.0 else "NG"))
df["D/C"] = df["D/C"].apply(lambda x: "-" if pd.isna(x) else f"{x:.2f}")
df

,load-path link,demand,capacity,D/C,status
0,1. wind pressure (LTS 3.8),36.6 psf on sign,-,-,-
1,2. sign U-bolt clamps,137 lb/leg,2237 lb,0.06,OK
2,3a. line post (bare fence),26.5 k-in,53.3 k-in,0.50,OK
3,"3b. sign post, sign low (design)",31.5 k-in,53.3 k-in,0.59,OK
4,3c. sign post if sign at fence top,51.3 k-in,53.3 k-in,0.96,OK
5,3d. post P+M interaction (5.12.1-3),sign low / at top,1.0,0.98,OK
6,4. fabric tension-band bolts,68 lb,2237 lb,0.03,OK
7,"5a. anchors, bare fence",6620 lb,concrete governs,0.93,OK
8,"5b. anchors, fence + low sign",7871 lb,concrete governs,1.10,NG
9,5c. wrap plate vs ODOT 7.1k,6.3 kips,7.1 kips,0.89,OK


## Conclusions

- **The standard VPF detail works for what it was designed for.**  Post,
  bands, and anchors all clear the bare-fence demand (the anchors at DCR ≈
  0.9 — no idle margin), and the wrap-plate demand sits at 75% of ODOT's
  published 7.1-kip design envelope.
- **Mounted low, the sign very nearly works on the standard detail:** the
  post, bands, and wrap-plate interface all pass; only the flat-plate
  anchorage's **bond limit state edges past 1.0** — and that on a
  conservatively clipped bond area.  A refined bond-area calc, BP-2's 5-ft
  post spacing, or a modestly smaller panel would close the gap.
- **Mounting height is the whole ballgame.**  Ride the same sign to the top
  of the fence and the standard post burns its entire flexural margin
  (interaction ≈ 1.0 even with the plastic section), fails fatigue below,
  the anchorage concrete DCR nearly doubles, and the wrap-plate demand blows
  44% past ODOT's envelope.  If the sign must sit high, it needs a special
  design: heavier post, anchor reinforcement tied into the parapet cage, or
  a dedicated support off the structure — more/larger rods won't help
  because concrete governs, and the VPF notes forbid substituting mechanical
  anchors.
- Left out of the calculation: The parapet itself and the deck overhang
  under the combined fence+sign line load (AASHTO LRFD BDS
  3.8.1.2.4 sound-barrier analogy, Section 13/A13.4), and the MASH
  crashworthiness question any barrier attachment raises.

**Fatigue note (LTS Section 11, Fatigue I):** Table 3.4-1 applies natural
wind gust, vortex shedding, and **galloping** each at 1.0, separately.  Two
apply here:

- *Natural wind gust* — $P_{NW} = 5.2\,C_d\,I_F$ (Eq. 11.7.1.2-1), horizontal
  on everything.
- *Galloping* — $P_G = 21\,I_F$ psf (Eq. 11.7.1.1-1), an equivalent static
  shear applied **vertically** over the sign panel's frontal area.  Strictly,
  11.7.1.1 targets attachments on cantilevered *horizontal* arms (C11.7.1.1:
  structures without such arms "are not susceptible"), but a solid panel
  rigidly clamped to a flexible cantilevered pipe is exactly the
  nonsymmetric-section situation galloping feeds on, so it is checked rather
  than argued away.  Fatigue stress ranges are elastic, so they ride on $S$,
  not $Z$.

**Fatigue resistance:** LTS-1 no longer hands the fillet-welded
tube-to-transverse-plate socket a blanket category.  Table 11.9.3.1-1
Detail 5.4 bands the CAFT by the infinite-life stress concentration factor
$K_I$ (Eq. 11.9.3.1-1, built on $K_F$ from Eq. 11.9.3.1-2): $K_I \le 4.0$ →
7.0 ksi, $4.0 < K_I \le 6.5$ → 4.5 ksi, $6.5 < K_I \le 7.7$ → 2.6 ksi.
$K_I$ cannot honestly be *computed* here — Eq. 11.9.3.1-2's validity window
($0.179 \le t_T \le 0.5$ in, $8 \le D_T \le 50$ in, $1.5 \le t_{TP} \le 4$
in, $1.25 \le C_{BC} \le 2.5$) misses this 2.880-in, 0.160-in-wall post on
a 1-in plate on **every parameter**; the NCHRP 10-70 calibration was for
signal poles and high-masts.  So the check below adopts the **worst band,
2.6 ksi, as a floor** rather than argue for a number the equation doesn't
cover.  (Physically the connection should do better — the 1-in plate is
enormously stiff relative to a 2.88-in tube, which is the flexibility that
drives $K_I$ up — but the conclusion doesn't need that argument.)

In [14]:
IF = 1.0
# natural wind gust (Eq. 11.7.1.2-1), horizontal ---------------------------
P_nw = 5.2 * Cd_sign * IF
sr_low = P_nw * A_sign * z_sign_cg     * 12 / S_p / 1000   # sign low (design)
sr_top = P_nw * A_sign * z_sign_cg_top * 12 / S_p / 1000   # sign at fence top
print(f"NWG:       P_NW = {P_nw:.1f} psf -> stress range: sign low {sr_low:.1f} ksi, "
      f"sign at top {sr_top:.1f} ksi")

# galloping (Eq. 11.7.1.1-1), vertical shear over the panel face -----------
P_g  = 21.0 * IF
F_g  = P_g * A_sign                        # lb, vertical, at the panel centroid
ecc  = OD/2 + 1.0                          # in, panel plane offset from post axis
sr_g = (F_g / A_p + F_g * ecc / S_p) / 1000
print(f"galloping: P_G = {P_g:.0f} psf -> F = {F_g:.0f} lb vertical at e = {ecc:.1f} in")
print(f"           post stress range = {sr_g:.1f} ksi (axial + weak-axis bending),")
print(f"           U-bolt legs {F_g/n_legs:.0f} lb each — both trivial: the panel hangs")
print("           off the post's strong direction, unlike a mast-arm tip")
caft_floor = 2.6                          # ksi, worst K_I band of Detail 5.4
print(f"CAFT, Detail 5.4: 7.0 / 4.5 / 2.6 ksi by K_I; K_I equations not valid for")
print(f"this geometry -> use the 2.6-ksi floor.  Anchor rods: Cat D, 7 ksi.")
print(f"-> sign low:    NWG {sr_low:.1f} ksi and galloping {sr_g:.1f} ksi (applied "
      f"separately) both < {caft_floor} ksi  OK")
print(f"-> sign at top: NWG {sr_top:.1f} ksi fails every band, even 7.0 — fatigue")
print("   condemns the top mount regardless of where K_I lands, and it does so")
print("   even where the strength check now squeaks by")

NWG:       P_NW = 6.1 psf -> stress range: sign low 1.9 ksi, sign at top 5.6 ksi
galloping: P_G = 21 psf -> F = 315 lb vertical at e = 2.4 in
           post stress range = 1.1 ksi (axial + weak-axis bending),
           U-bolt legs 79 lb each — both trivial: the panel hangs
           off the post's strong direction, unlike a mast-arm tip
CAFT, Detail 5.4: 7.0 / 4.5 / 2.6 ksi by K_I; K_I equations not valid for
this geometry -> use the 2.6-ksi floor.  Anchor rods: Cat D, 7 ksi.
-> sign low:    NWG 1.9 ksi and galloping 1.1 ksi (applied separately) both < 2.6 ksi  OK
-> sign at top: NWG 5.6 ksi fails every band, even 7.0 — fatigue
   condemns the top mount regardless of where K_I lands, and it does so
   even where the strength check now squeaks by


---
### References

- AASHTO *LRFD Specifications for Structural Supports for Highway Signs,
  Luminaires, and Traffic Signals* (LRFDLTS-1), 1st Ed. w/ interims — Arts.
  3.4, 3.8, 5.8.2, 5.10, 5.12.1, 5.15, 5.16, 11.7, 11.9.3.1
- ODOT SCD **VPF-1-24** (2024) and **VPF-1-90** (1990, rev. 2011) — Vandal
  Protection Fence
- ICC-ES **ESR-3187** — Hilti HIT-HY 200 adhesive anchors (Tables 10, 12, 14)
- ACI 318-19 Ch. 17 via `civilpy.structural.concrete.AnchorBolts`
- CLFMI *Wind Load Guide* — chain-link fabric net-area treatment